# MSH chain on a 2D substrate: tensor networks

**Goal.** Measure how the cost of the tensor-network (MPO/MPS) methods grows with system size, against the dense and sparse
Python methods of `msh_substrate_python.ipynb`, for an MSH chain on a 2D superconducting substrate. The substrate is grown first in
width (more substrate around the same chain), then as a whole.

**Model.** A $2^{L_x}\times2^{L_y}$ square substrate with hopping $t = 1$, chemical potential $\mu = -7$, Rashba $\alpha = 0.9$ and
s-wave pairing $\Delta = 2.4$ on every site. The magnetic chain is the centred segment of $\ell$ sites in the middle row
$i_y = N_y/2$, with exchange $J\,\mathbf S\cdot\boldsymbol\sigma$, $J = 5.8$. With $L_y = 0$ (one row) this is exactly the MSH
chain of `msh_chain_tn.ipynb` with a centred topological segment. At $\mu = -7$ the bare substrate band is empty, so the substrate
is a gapped, trivial superconductor; the chain stays topological on it (`msh_substrate_python.ipynb`, section 3). At this point the
zero modes barely enter the substrate (gap $\approx 0.55$, decay within one row), which favours the tensor network. Section 8 repeats
the benchmark at a second point, $\Delta = 0.6$, $\mu = -3$ (inside the band), $J = 3$, where they spread over several rows.

**Code.** The model (`Substrate`, spin textures, `msh_substrate`), the MPS/MPO helpers, the zero-mode pipeline of section 6 and
the second parameter point `PC` are in `substrate_model.jl`, included in the setup cell; the Python model is in
`substrate_model.py`. Both are shared with the time-dependent notebooks. This notebook keeps the checks, the benchmarks and the
tables; the markdown of sections 2, 3 and 6 still explains the code.

**Contents**
1. Setup and helpers
2. Geometry
3. Hamiltonian MPO
4. Check data for the Python notebook
5. MPO size and build time against system size
6. Zero-mode pipeline
7. Zero-mode benchmark
8. A substrate the zero modes leak into ($\Delta = 0.6$, $\mu = -3$, $J = 3$)
9. MPO cutoff and the largest substrates
10. Chain length

**Conventions.** Sites `[nambu; spin; y₁ … y_Ly; x₁ … x_Lx]`, most significant bit first, so the position index is
$n = i_x + i_y N_x$ (TensorBinding's 2D row-major convention). Nambu basis $(c_\uparrow, c_\downarrow, c^\dagger_\uparrow, c^\dagger_\downarrow)$,
as in the other Julia notebooks; the Python notebook uses kwant's $(c_\uparrow, c_\downarrow, c^\dagger_\downarrow, -c^\dagger_\uparrow)$.

## 1. Setup and helpers

In [ ]:
using TensorBinding, ITensors, ITensorMPS, LinearAlgebra, Printf, NPZ, Random, Logging, DelimitedFiles
BLAS.set_num_threads(1)            # single thread, as in the Python notebook, so that timings compare like with like

const DATA_DIR = "data"            # check data for msh_substrate_python.ipynb (not committed)
mkpath(DATA_DIR)

include("substrate_model.jl")   # model, MPO helpers and zero-mode pipeline (shared with the time-dependent notebooks)

site_kpm_moments (generic function with 1 method)

## 2. Geometry

`Substrate(Lx, Ly, ℓ)`: $N_x = 2^{L_x}$ columns, $N_y = 2^{L_y}$ rows, and the chain of $\ell$ sites centred in row $N_y/2$.
`chain_x` lists the (0-indexed) columns of the chain sites.

In [ ]:
# Substrate, Nx, Ny, nsites, chain_row, chain_x, S_z and S_tilt are in substrate_model.jl
g = Substrate(6, 5, 48)
@printf("Substrate(6, 5, 48): %d × %d sites, chain row %d, columns %d … %d\n", Nx(g), Ny(g), chain_row(g), first(chain_x(g)), last(chain_x(g)))

Substrate(6, 5, 48): 64 × 32 sites, chain row 16, columns 8 … 55


## 3. Hamiltonian MPO

Particle block
$$H_0 = \sum_r c_r^\dagger(-\mu)c_r + \sum_r\bigl[c_{r+\hat x}^\dagger h_x c_r + c_{r+\hat y}^\dagger h_y c_r + \text{h.c.}\bigr]
+ J\sum_{r\in\text{chain}} c_r^\dagger\,\mathbf S(r)\cdot\boldsymbol\sigma\,c_r,\qquad
h_x = -t + i\alpha\sigma_y,\quad h_y = -t - i\alpha\sigma_x,$$
hole block $-H_0^*$ and singlet pairing $\Delta\,i\sigma_y$, assembled as in `msh_chain` of `msh_chain_tn.ipynb`.

- **x bonds:** the open binary shift $|n\rangle\to|n+1\rangle$ on all position qubits, multiplied by a mask that removes the last
  column, so that $i_x = N_x - 1$ does not hop to $i_x = 0$ of the next row. The mask is the same on every row:
  $(1,1)^{\otimes L_y}\otimes m(i_x)$ with $m = 0$ at $i_x = N_x-1$ and 1 elsewhere.
- **y bonds:** the open shift $|n\rangle\to|n+N_x\rangle$, which maps the top row to zero (open boundary). With one row
  ($L_y = 0$) there are no y bonds; `shift_pair_mpos(pos, Nx)` would then keep only the low bits of $N_x = 2^{L}$, i.e. act as the
  identity, so it must not be called.
- **Exchange:** one diagonal MPO per spin component, $e_{i_{y0}}\otimes JS_k(i_x)$: nonzero on the chain row only, with the chain profile
  on the x qubits.

Both diagonals are built with `row_profile_mps` (a product state on the y qubits times a compressed profile of length $N_x$ on the x
qubits) and `diag_mpo`, so building $H$ costs $O(N_x)$, not $O(N)$. (An earlier version compressed full length-$N$ vectors with
`diag_mpo_exact`; at $512\times512$ and $1024\times1024$ sites that took 12–14 s of the build.)

In [ ]:
# msh_substrate is in substrate_model.jl
HB, sites = msh_substrate(Substrate(3, 2, 5), S_z)
@printf("msh_substrate(Substrate(3, 2, 5), S_z): %d sites, χ(H) = %d\n", length(sites), maxlinkdim(HB))

msh_substrate(Substrate(3, 2, 5), S_z): 7 sites, χ(H) = 22


## 4. Check data for the Python notebook

Dense matrices of the MPO for ten small systems (one row; two to eight rows; a substrate taller than wide; the $\hat z$ and the tilted
texture), written to `data/tn_check_H.npz`. `msh_substrate_python.ipynb` (section 2) compares them element by element with its sparse
Hamiltonian after the change of Nambu basis.

In [ ]:
check_cases = [(3, 0, 6), (3, 1, 6), (3, 2, 5), (2, 3, 3), (4, 2, 10)]
check = Dict{String,Matrix{ComplexF64}}()
for (Lx, Ly, ℓ) in check_cases, (name, S) in (("z", S_z), ("tilt", S_tilt))
    HB, sites = msh_substrate(Substrate(Lx, Ly, ℓ), S)
    H = dense(HB, sites)
    check["H_$(Lx)_$(Ly)_$(ℓ)_$name"] = H
    @printf("Lx = %d, Ly = %d, ℓ = %2d, %-4s: D = %3d, χ(H) = %d, ‖H − H†‖ = %.1e\n", Lx, Ly, ℓ, name, size(H, 1), maxlinkdim(HB), norm(H - H'))
end
npzwrite(joinpath(DATA_DIR, "tn_check_H.npz"), check)
println("wrote ", joinpath(DATA_DIR, "tn_check_H.npz"))

Lx = 3, Ly = 0, ℓ =  6, z   : D =  32, χ(H) = 15, ‖H − H†‖ = 3.1e-14
Lx = 3, Ly = 0, ℓ =  6, tilt: D =  32, χ(H) = 15, ‖H − H†‖ = 7.3e-14


Lx = 3, Ly = 1, ℓ =  6, z   : D =  64, χ(H) = 22, ‖H − H†‖ = 1.7e-13
Lx = 3, Ly = 1, ℓ =  6, tilt: D =  64, χ(H) = 24, ‖H − H†‖ = 2.2e-13


Lx = 3, Ly = 2, ℓ =  5, z   : D = 128, χ(H) = 22, ‖H − H†‖ = 1.1e-13
Lx = 3, Ly = 2, ℓ =  5, tilt: D = 128, χ(H) = 24, ‖H − H†‖ = 3.5e-13


Lx = 2, Ly = 3, ℓ =  3, z   : D = 128, χ(H) = 22, ‖H − H†‖ = 1.3e-13
Lx = 2, Ly = 3, ℓ =  3, tilt: D = 128, χ(H) = 26, ‖H − H†‖ = 1.7e-13


Lx = 4, Ly = 2, ℓ = 10, z   : D = 256, χ(H) = 21, ‖H − H†‖ = 2.3e-12
Lx = 4, Ly = 2, ℓ = 10, tilt: D = 256, χ(H) = 36, ‖H − H†‖ = 2.0e-13


wrote data/tn_check_H.npz

## 5. MPO size and build time against system size

Width scan ($L_x = 6$, $L_y = 0\dots6$, $\ell = 48$) and square scan ($L_x = L_y = 4\dots11$, $\ell = 3N_x/4$, up to
$2048\times2048$ sites). The bond dimension of the Hamiltonian, $\chi(H)$, and its memory (number of stored elements of the MPO tensors)
are compared with the number of nonzeros of the sparse matrix (`msh_substrate_python.ipynb`, section 4). Build times are measured after
a warm-up build of each size, so they do not include compilation; with the $O(N_x)$ construction of section 3 they stay in the
millisecond range at every size.

In [ ]:
# mpo_size is in substrate_model.jl
scan = vcat([(6, Ly, 48) for Ly in 0:6], [(L, L, 3 * 2^L ÷ 4) for L in 4:11])
build = []
for (Lx, Ly, ℓ) in scan
    g = Substrate(Lx, Ly, ℓ)
    msh_substrate(g, S_z)                                        # warm-up
    t_build = @elapsed (HB = first(msh_substrate(g, S_z)))
    push!(build, (; Lx, Ly, ℓ, N = nsites(g), χ = maxlinkdim(HB), mem = mpo_size(HB), t_build))
    @printf("Lx = %d, Ly = %d: N = %6d sites, D = %7d, χ(H) = %d, MPO elements = %6d, build %.2f s\n",
            Lx, Ly, nsites(g), 4nsites(g), maxlinkdim(HB), mpo_size(HB), t_build)
end

Lx = 6, Ly = 0: N =     64 sites, D =     256, χ(H) = 20, MPO elements =   3348, build 0.04 s
Lx = 6, Ly = 1: N =    128 sites, D =     512, χ(H) = 32, MPO elements =   4584, build 0.03 s


Lx = 6, Ly = 2: N =    256 sites, D =    1024, χ(H) = 32, MPO elements =   8820, build 0.02 s
Lx = 6, Ly = 3: N =    512 sites, D =    2048, χ(H) = 22, MPO elements =  10416, build 0.02 s
Lx = 6, Ly = 4: N =   1024 sites, D =    4096, χ(H) = 32, MPO elements =  15960, build 0.03 s
Lx = 6, Ly = 5: N =   2048 sites, D =    8192, χ(H) = 32, MPO elements =  12980, build 0.03 s


Lx = 6, Ly = 6: N =   4096 sites, D =   16384, χ(H) = 29, MPO elements =  17072, build 0.03 s
Lx = 4, Ly = 4: N =    256 sites, D =    1024, χ(H) = 24, MPO elements =   5932, build 0.01 s
Lx = 5, Ly = 5: N =   1024 sites, D =    4096, χ(H) = 31, MPO elements =  13808, build 0.03 s
Lx = 6, Ly = 6: N =   4096 sites, D =   16384, χ(H) = 29, MPO elements =  17072, build 0.03 s


Lx = 7, Ly = 7: N =  16384 sites, D =   65536, χ(H) = 29, MPO elements =  19568, build 0.03 s
Lx = 8, Ly = 8: N =  65536 sites, D =  262144, χ(H) = 23, MPO elements =  15756, build 0.07 s


Lx = 9, Ly = 9: N = 262144 sites, D = 1048576, χ(H) = 32, MPO elements =  27312, build 0.05 s
Lx = 10, Ly = 10: N = 1048576 sites, D = 4194304, χ(H) = 32, MPO elements =  31552, build 0.05 s


Lx = 11, Ly = 11: N = 4194304 sites, D = 16777216, χ(H) = 32, MPO elements =  25500, build 0.05 s


## 6. Zero-mode pipeline

The pipeline of `msh_chain_tn.ipynb` (sections 8–9), adapted to the substrate:

1. `zero_mode_subspace`: penalized DMRG on $K = H^2 + \eta^2$ ($\eta = 10^{-2}$) for the two near-zero states and a third state as a gap
   check; $E_{\rm gap} = \lVert H\psi_3\rVert$ should sit at the bulk gap. The time of building $K$ and of the three DMRG runs is
   returned separately. `cutoff_K = 1e-20`: see below. **Start of the second state** (below).
2. `fermion_from_subspace`: localize with the position operator $\hat X = \sum_k 2^{L_x-k}\hat n_k$ on the **x qubits only** (in 1D
   all position qubits are x qubits), fix $C\gamma = \gamma$, pair into $\psi_f = (\gamma_1 + i\gamma_2)/\sqrt2$ with $E_f > 0$.
3. $\Delta Q = \langle\psi_f|\tau_z|\psi_f\rangle$.

The benchmark uses $\ell = 12$: there $E_f \approx 4\times10^{-4}$ and $|\Delta Q| \approx 3\times10^{-3}$ on the substrate, both far above
round-off and the DMRG accuracy, so they measure accuracy. (For $\ell = 48$ the splitting is $\sim10^{-14}$ and $\Delta Q$ would be pure
round-off.) With $E_f \gg 2\times10^{-6}$ no inverse-iteration refinement is needed (`msh_chain_tn.ipynb`, section 9).

**Truncation of $K$.** $\Delta Q$ is first order in any admixture of bulk states in the pair (energies only second order), so it is the
quantity most sensitive to how $K$ is built. With `cutoff_K = 1e-14` the one-row chain ($L_y = 0$) came out with
$|\Delta Q_{\rm TN} - \Delta Q_{\rm exact}| = 1.3\times10^{-5}$: the pair states carried a weight $4.7\times10^{-10}$ outside the exact pair,
independent of the number of sweeps and of the DMRG cutoff, and gone with `cutoff_K = 1e-20` ($1.3\times10^{-10}$). With a substrate the
error was a few $\times10^{-9}$ for either value, limited by DMRG convergence. The benchmark therefore uses `cutoff_K = 1e-20` (as the
$\Delta Q$ calculations of `msh_chain_tn.ipynb` do); it doubles $\chi(K)$ (18 → 38) and makes the DMRG 1.2–1.6 times slower on the
large substrates, which is part of the cost of the method at this accuracy. The inverse-iteration step of `msh_chain_tn.ipynb` is not
an alternative here: with $E_f$ close to $\sigma = 10^{-3}$ it made $L_y = 1$ worse ($5.7\times10^{-5}$).

**Start of the second state.** With a random start, the DMRG for the second zero mode can stop in a wrong state: at the second
parameter point of section 8 and $2048\times2048$ sites it ended at $\lVert H\psi_2\rVert = 0.60$ (a substrate band-edge state, bond dimension
at the cap of 200), and the gap check picked up the missing zero mode instead. The first state found by DMRG is often, but not always, a single
localized Majorana. Two better starts for the second state:

- $C\psi_1$, the particle–hole conjugate (the near-zero subspace is closed under $C$), orthogonalized to $\psi_1$. It is useless when
  $\psi_1$ is itself self-conjugate (a single Majorana): then the part of $C\psi_1$ orthogonal to $\psi_1$ has norm $0.000$.
- The mirror image of $\psi_1$, $x \to N_x - 1 - x$, which maps one end of the centred chain onto the other. In the quantics encoding it is the
  bitwise complement of $i_x$: a NOT gate on every x qubit (`mirror_x`). Orthogonalized to $\psi_1$ it keeps norm $0.98$–$1.00$.

`zero_mode_subspace(…; xq)` starts the second state from the mirror image (`xq` = site numbers of the x qubits), and falls back to
$C\psi_1$ (if its orthogonal part has norm above 0.1) and then to a random start; it returns which start it used. With this start the
$2048\times2048$ case of section 8 converges to the exact values.

**Which basis of the pair DMRG returns, and its cost.** The near-zero subspace is two-dimensional, and DMRG can return any orthonormal
basis of it: two localized Majoranas, or two-ended mixtures like $\psi_f$. The two differ in cost. For the exact modes at the second
parameter point (TT-SVD with the DMRG cutoff $10^{-14}$) each Majorana needs $\chi = 121$ and $\psi_f$ needs $139$, independent of the
system size and of where the chain sits on the binary grid (shifts of 0–16 sites at $256^2$ and $512^2$). Which basis comes out is set by
the random start of $\psi_1$ and varies from size to size: in section 8, $\psi_1$ is a single Majorana at $128\times64$, $512^2$ and $2048^2$
($\chi \approx 114$; the `C_perp` column of `tn_candidate.csv` is $\approx 0$) and a mixture at $256^2$ and $1024^2$ ($\chi = 137$, $135$). Since
the DMRG cost grows as $\chi^3$, this makes the time of a size uncertain by up to $(137/114)^3 \approx 1.7$; it is why $1024^2$ takes longer
than $2048^2$ there. The mirror start of $\psi_2$ does not change it (it follows $\psi_1$), and $C\psi_1$ would steer to the two-ended
basis whenever $\psi_1$ is a mixture. Steering $\psi_1$ was tried and did not help: starting it from a random MPS projected onto the
left half of the substrate gave a Majorana at $128\times64$ and $256^2$ ($\chi = 112$, $120$), but at $512^2$, $1024^2$ and $2048^2$ the DMRG
still converged to mixtures ($\chi = 136$–$137$; `C_perp` $0.30$, $0.63$, $0.98$), and the scan took 5 % longer in total.


In [ ]:
# quietly, norm_Hψ, apply_site, τz_on, ΔQ_mode, mps_size, xqubits, Cconj, mirror_x, zero_mode_subspace and
# fermion_from_subspace are in substrate_model.jl
println("zero-mode pipeline: ", join(string.((zero_mode_subspace, fermion_from_subspace)), ", "))

zero-mode pipeline: zero_mode_subspace, fermion_from_subspace


## 7. Zero-mode benchmark

For every size of the width scan ($L_x = 6$, $L_y = 0\dots6$) and of the square scan ($L_x = L_y = 4\dots11$, up to $2048\times2048$
sites; $L = 6$ is the last point of the width scan), all with $\ell = 12$ and spins along $\hat z$: build $H$ and $K$, run the three
DMRGs, localize and pair. The sizes run in order of increasing $N$. Recorded: the times of each step (single thread, after a warm-up
run on a small system), the bond dimensions $\chi(K)$, $\chi(\psi)$, the memory of $K$ and of the three MPS (stored elements × 16
bytes), the peak resident memory of the Julia process after the size (`Sys.maxrss()`; it includes the Julia runtime and compiled
code, and since $N$ increases it is the peak up to and including that size), $E_f$, $\Delta Q$, the gap estimate and the Majorana
centres.

Results go to `tn_zero_modes.csv` (committed, read by the comparison notebook). With `RUN_ZM = false` the cell loads that file instead
of recomputing; progress of a run is appended to `data/zm_progress.log`. The second DMRG state starts from the mirror image of the first
(section 6); the last column of the table says which start was used.

In [ ]:
RUN_ZM = true
zm_file = "tn_zero_modes.csv"
zm_cols = ["Lx", "Ly", "ell", "N", "D", "t_H", "t_K", "t_dmrg", "t_pair", "t_total", "chi_H", "chi_K", "chi_psi",
           "mem_K_MB", "mem_psi_MB", "rss_MB", "E_f", "dQ", "E_gap", "x1", "x2", "start2"]
zm_scan = sort(vcat([(6, Ly) for Ly in 0:6], [(L, L) for L in (4, 5, 7, 8, 9, 10, 11)]); by = p -> (sum(p), p[2]))   # increasing N

function zm_run(Lx, Ly; ℓ=12)
    g = Substrate(Lx, Ly, ℓ)
    t_H = @elapsed ((HB, sites) = msh_substrate(g, S_z))
    z = zero_mode_subspace(HB, sites; xq = xqubits(g))
    t_pair = @elapsed (r = fermion_from_subspace(HB, sites, z.ψs, g))
    dQ = ΔQ_mode(r.ψf)
    return Any[Lx, Ly, ℓ, nsites(g), 4nsites(g), t_H, z.t_K, z.t_dmrg, t_pair, t_H + z.t_K + z.t_dmrg + t_pair,
            maxlinkdim(HB), maxlinkdim(z.K), maximum(maxlinkdim, z.ψs), 16e-6 * mpo_size(z.K),
            16e-6 * sum(mps_size, z.ψs), Sys.maxrss() / 1e6, r.E_f, dQ, z.E_gap, r.centers..., z.start2[1]]
end

zm = if !RUN_ZM && isfile(zm_file)
    println("loading ", zm_file); readdlm(zm_file, ','; header=true)[1]
else
    zm_run(4, 0)                                                   # warm-up (compilation)
    rows = []
    open(joinpath(DATA_DIR, "zm_progress.log"), "w") do log
        for (Lx, Ly) in zm_scan
            row = zm_run(Lx, Ly); push!(rows, row)
            println(log, join(row, ", ")); flush(log)
        end
    end
    M = permutedims(reduce(hcat, rows))
    open(zm_file, "w") do io; println(io, join(zm_cols, ",")); writedlm(io, M, ','); end
    M
end

@printf("%3s %3s %8s %9s %6s %6s %7s %6s %4s %5s %5s %8s %8s %13s %13s %7s  %s\n", "Lx", "Ly", "N", "D", "t_K", "DMRG", "pair",
        "total", "χ_H", "χ_K", "χ_ψ", "K+ψ MB", "RSS MB", "E_f", "ΔQ", "gap", "start of ψ₂")
for r in eachrow(zm)
    @printf("%3d %3d %8d %9d %5.1fs %5.0fs %6.1fs %5.0fs %4d %5d %5d %8.2f %8.0f %13.6e %+13.6e %7.4f  %s\n", r[1], r[2], r[4], r[5], r[7],
            r[8], r[9], r[10], r[11], r[12], r[13], r[14] + r[15], r[16], r[17], r[18], r[19], r[22])
end

 Lx  Ly        N         D    t_K   DMRG    pair  total  χ_H   χ_K   χ_ψ   K+ψ MB   RSS MB           E_f            ΔQ     gap  start of ψ₂
  6   0       64       256   0.0s     3s    0.0s     3s   20    29     7     0.14     1402  1.491239e-03 +4.159299e-02  0.3056  mirror


  6   1      128       512   0.0s     9s    0.0s     9s   32    35    16     0.22     1490  1.868688e-04 -2.135578e-03  0.4398  mirror
  6   2      256      1024   0.0s    13s    0.0s    13s   30    38    21     0.28     1502  4.146924e-04 -3.863993e-03  0.5639  mirror
  4   4      256      1024   0.0s    13s    0.2s    13s   24    26    26     0.24     1553  3.715561e-04 -3.441197e-03  0.5529  mirror
  6   3      512      2048   0.1s    21s    0.0s    21s   32    37    21     0.37     1553  3.713820e-04 -3.448271e-03  0.5529  mirror
  6   4     1024      4096   0.1s    37s    0.0s    37s   32    37    24     0.45     1560  3.712285e-04 -3.446598e-03  0.5529  mirror
  5   5     1024      4096   0.1s    30s    0.0s    30s   32    38    24     0.42     1560  3.712285e-04 -3.446599e-03  0.5529  mirror
  6   5     2048      8192   0.0s    34s    0.2s    34s   30    38    27     0.46     1566  3.712285e-04 -3.446600e-03  0.5529  mirror
  6   6     4096     16384   0.1s    35s    0.0s    35s

## 8. A substrate the zero modes leak into ($\Delta = 0.6$, $\mu = -3$, $J = 3$)

At the point of section 7 the zero modes stay on the chain: $\mu = -7$ lies below the band ($\pm4.74$ with Rashba), and $\Delta = 2.4$ makes
the coherence length shorter than a site, so only $\sim2\times10^{-5}$ of the weight lies 4 or more rows away. A μ × J × Δ scan
(`msh_substrate_python.ipynb`, section 8) finds the topological region and how far the zero mode spreads; leakage needs both $\mu$ in the
band and a smaller $\Delta$. The point $\Delta = 0.6$, $\mu = -3$, $J = 3$ (with $t = 1$, $\alpha = 0.9$) is topological, gapped (gap
$0.0902$, six times smaller) and the mode spreads over several rows: 0.32 of its weight on the chain row, 0.13 four or more rows
away, decaying as $e^{-|\Delta y|/2}$. The pair splitting decays along the chain with a length of about 7 sites, so the chain is
$\ell = 64$ ($E_f/E_{\rm gap} = 10^{-4}$, comparable to $\ell = 12$ in section 7); the exact values are $E_f = 9.04771\times10^{-6}$,
$\Delta Q = 4.58344\times10^{-4}$ (converged from $128\times32$ sites on).

The same pipeline (section 6, with the mirror start) on $128\times64$ and the square substrates $256^2$ to $2048^2$. Results go to
`tn_candidate.csv` (committed; compared with MUMPS in `msh_substrate_scaling.ipynb`, section 6), one row as soon as it is done, and
sizes already in the file are skipped, so an interrupted run resumes; `RUN_CAND = false` only loads the file. About 2.5 hours.
`rss_MB` is the peak of the Julia process so far, so it includes section 7.

In [ ]:
RUN_CAND = true
cand_file = "tn_candidate.csv"
# const PC = (μ = -3.0, Δ = 0.6, J = 3.0) is in substrate_model.jl
cand_scan = [(7, 6), (8, 8), (9, 9), (10, 10), (11, 11)]
cand_cols = ["Lx", "Ly", "ell", "N", "t_H", "t_K", "t_dmrg", "t_pair", "t_total", "chi_K", "chi_psi", "mem_K_MB", "mem_psi_MB",
             "rss_MB", "E_f", "dQ", "E_gap", "E_sub1", "E_sub2", "x1", "x2", "start2", "C_perp", "mirror_perp"]

function cand_run(Lx, Ly; ℓ=64)
    g = Substrate(Lx, Ly, ℓ)
    t_H = @elapsed ((HB, sites) = msh_substrate(g, S_z; PC...))
    z = zero_mode_subspace(HB, sites; xq = xqubits(g))
    t_pair = @elapsed (r = fermion_from_subspace(HB, sites, z.ψs, g))
    return Any[Lx, Ly, ℓ, nsites(g), t_H, z.t_K, z.t_dmrg, t_pair, t_H + z.t_K + z.t_dmrg + t_pair, maxlinkdim(z.K),
               maximum(maxlinkdim, z.ψs), 16e-6 * mpo_size(z.K), 16e-6 * sum(mps_size, z.ψs), Sys.maxrss() / 1e6,
               r.E_f, ΔQ_mode(r.ψf), z.E_gap, z.E_sub..., r.centers..., z.start2...]
end

done = isfile(cand_file) ? Set((Int(r[1]), Int(r[2])) for r in eachrow(readdlm(cand_file, ','; header=true)[1])) : Set{Tuple{Int,Int}}()
if RUN_CAND
    isempty(done) && open(io -> println(io, join(cand_cols, ",")), cand_file, "w")
    for (Lx, Ly) in cand_scan
        (Lx, Ly) in done && continue
        row = cand_run(Lx, Ly)
        open(io -> println(io, join(row, ",")), cand_file, "a")
        @printf("done %d×%d: %.0f s\n", 2^Lx, 2^Ly, row[9]); flush(stdout)
    end
end

cand = readdlm(cand_file, ','; header=true)[1]
@printf("%11s %8s %7s %4s %4s %7s %13s %13s %8s %10s %10s  %s\n", "substrate", "N", "total", "χ_K", "χ_ψ", "RSS MB", "E_f", "ΔQ",
        "gap", "‖Hψ₂‖", "x₁ … x₂", "start of ψ₂ (C⊥, mirror⊥)")
for r in eachrow(cand)
    @printf("%5d×%-5d %8d %6.0fs %4d %4d %7.0f %13.6e %+13.6e %8.5f %10.2e %4.0f … %4.0f  %s (%.3f, %.3f)\n", 2^r[1], 2^r[2], r[4],
            r[9], r[10], r[11], r[14], r[15], r[16], r[17], r[19], r[20], r[21], r[22], r[23], r[24])
end

done 128×64: 395 s


done 256×256: 1141 s


done 512×512: 1802 s


done 1024×1024: 2847 s


done 2048×2048: 2424 s


  substrate        N   total  χ_K  χ_ψ  RSS MB           E_f            ΔQ      gap      ‖Hψ₂‖    x₁ … x₂  start of ψ₂ (C⊥, mirror⊥)
  128×64        8192    395s   30  113    1869  9.047711e-06 +4.583427e-04  0.09025   9.06e-06   33 …   94  mirror (0.001, 1.000)


  256×256      65536   1141s   31  138    3045  9.047709e-06 +4.583436e-04  0.09025   9.06e-06   97 …  158  mirror (0.312, 1.000)
  512×512     262144   1802s   37  137    3885  9.047709e-06 +4.583435e-04  0.09025   9.07e-06  225 …  286  mirror (0.435, 1.000)
 1024×1024   1048576   2847s   38  137    3885  9.047709e-06 +4.583438e-04  0.09025   9.08e-06  481 …  542  mirror (0.895, 1.000)
 2048×2048   4194304   2424s   38  114    3885  9.047709e-06 +4.583434e-04  0.09025   9.08e-06  993 … 1054  mirror (0.000, 1.000)


## 9. MPO cutoff and the largest substrates

`msh_substrate` sums the terms of $H$ with ITensor's `+(…; cutoff)`, which truncates each bond of the sum to a *relative* discarded
weight `cutoff`. The norm of $H$ grows with the lattice, so the absolute error of a fixed relative cutoff grows with $N$. Until
4 November 2026 the default was $10^{-13}$: $\chi(H) = 7$–9 where the exact sum needs about 32, harmless at $256^2$ but an error in
$H\psi$ of $10^{-10}$–$10^{-6}$ from $1024^2$ on (`msh_substrate_td_tn.ipynb` shows its effect on TDVP). The default is now $10^{-20}$
for the static Hamiltonian; `ramp_mpo` uses $10^{-16}$, enough for the ramps.

The scan: the zero-mode pipeline of section 7 at the benchmark point with cutoff $10^{-13}$, $10^{-16}$ and $10^{-20}$, up to
$16384^2$ ($N = 2.7\times10^8$). Reference: MUMPS at $1024^2$ (`py_zero_modes.csv`); $E_f$ and $\Delta Q$ do not depend on the
substrate size beyond $256^2$ (MUMPS: to $10^{-12}$), so the same reference serves every size. Results in `tn_mpo_cutoff.csv`; the
times were measured with other runs on the machine (`RUN_BIG = false` only loads).

In [ ]:
RUN_BIG = false
big_file = "tn_mpo_cutoff.csv"
big_scan = vcat([(L, 1e-13) for L in (10, 11)], [(L, 1e-16) for L in 8:14], [(L, 1e-20) for L in 11:14])

function big_run(L, cutoff)
    g = Substrate(L, L, 12)
    t_H = @elapsed ((HB, sites) = msh_substrate(g, S_z; cutoff))
    z = zero_mode_subspace(HB, sites; xq = xqubits(g))
    r = fermion_from_subspace(HB, sites, z.ψs, g)
    return Any[cutoff, L, L, nsites(g), maxlinkdim(HB), maxlinkdim(z.K), maximum(maxlinkdim, z.ψs), t_H, z.t_K, z.t_dmrg, r.E_f,
               ΔQ_mode(r.ψf), z.E_gap]
end

if RUN_BIG
    open(big_file, "w") do io
        println(io, "cutoff,Lx,Ly,N,chi_H,chi_K,chi_psi,t_H,t_K,t_dmrg,E_f,dQ,E_gap")
        for (L, c) in big_scan
            println(io, join(big_run(L, c), ",")); flush(io)
        end
    end
end

B = readdlm(big_file, ','; header = true)[1]
K_ = readdlm("py_zero_modes.csv", ','; header = true); kc = Dict(c => i for (i, c) in enumerate(vec(K_[2])))
kref = K_[1][findfirst(r -> r[kc["Lx"]] == 10 && r[kc["Ly"]] == 10 && r[kc["method"]] == "mumps", collect(eachrow(K_[1]))), :]
Eref, Qref = kref[kc["E_f"]], kref[kc["dQ"]]
@printf("%7s %13s %5s %5s %7s %16s %11s %11s\n", "cutoff", "substrate", "χ_H", "χ_ψ", "DMRG", "E_f", "rel. error", "ΔQ error")
for r in eachrow(B)
    @printf("%7.0e %6d×%-6d %5d %5d %6.0fs %16.10e %11.1e %11.1e\n", r[1], 2^r[2], 2^r[3], r[5], r[7], r[10], r[11],
            abs(r[11] - Eref) / Eref, abs(r[12] - Qref))
end

 cutoff     substrate   χ_H   χ_ψ    DMRG              E_f  rel. error    ΔQ error
  1e-13   1024×1024       7    24    141s 3.7122848597e-04     2.9e-08     2.4e-09


  1e-13   2048×2048       7    24    188s 3.7122846056e-04     3.9e-08     2.6e-09
  1e-16    256×256       16    24    106s 3.7122847515e-04     1.7e-10     2.5e-09
  1e-16    512×512        9    24    129s 3.7122847542e-04     8.8e-10     1.9e-09
  1e-16   1024×1024       8    24    113s 3.7122847089e-04     1.1e-08     1.7e-09
  1e-16   2048×2048      16    24     94s 3.7122847831e-04     8.7e-09     1.5e-09
  1e-16   4096×4096      16    27    204s 3.7122843881e-04     9.8e-08     6.7e-09
  1e-16   8192×8192      11    26    282s 3.7041241302e-04     2.2e-03     5.7e-04
  1e-16  16384×16384      8    24    289s 3.7122786915e-04     1.6e-06     9.0e-07
  1e-20   2048×2048      32    27   2168s 3.7122846901e-04     1.6e-08     2.2e-10
  1e-20   4096×4096      31    26    596s 3.7122847350e-04     4.3e-09     2.4e-09
  1e-20   8192×8192      32    26    221s 3.7122849033e-04     4.1e-08     5.1e-09
  1e-20  16384×16384     32    24    223s 3.7122862168e-04     3.9e-07     1.3e-08


## 10. Chain length

Does the bond dimension grow with the length $\ell$ of the chain? The zero-mode pipeline on $512^2$ at both parameter points, $\ell = 8$ to
128 ($\ell \ge 16$ at the leaking point), and on $1024^2$ for $\ell = 64$, 128 at the benchmark point. These runs used the MPO cutoff
$10^{-16}$ (the default at the time; at $512^2$ and $1024^2$ it gives the same zero modes as $10^{-20}$, section 9) and shared the machine,
so their times are only indicative. Results in `ell_scan_static.csv` (`RUN_ELL = false` only loads).

In [ ]:
RUN_ELL = false
ell_file = "ell_scan_static.csv"
ell_scan = vcat([("bench", 9, ℓ) for ℓ in (8, 12, 16, 24, 32, 48, 64, 96, 128)], [("bench", 10, ℓ) for ℓ in (64, 128)],
                [("cand", 9, ℓ) for ℓ in (16, 24, 32, 48, 64, 96, 128)])

function ell_run(pt, L, ℓ)
    g = Substrate(L, L, ℓ); P = pt == "bench" ? NamedTuple() : PC
    t_H = @elapsed ((HB, sites) = msh_substrate(g, S_z; P...))
    z = zero_mode_subspace(HB, sites; xq = xqubits(g))
    t_pair = @elapsed (r = fermion_from_subspace(HB, sites, z.ψs, g))
    return Any[pt, L, ℓ, maxlinkdim(HB), maxlinkdim(z.K), maximum(maxlinkdim, z.ψs), t_H, z.t_K, z.t_dmrg, t_pair, r.E_f,
               ΔQ_mode(r.ψf), z.E_gap, r.centers...]
end

if RUN_ELL
    open(ell_file, "w") do io
        println(io, "point,L,ell,chi_H,chi_K,chi_psi,t_H,t_K,t_dmrg,t_pair,E_f,dQ,E_gap,x1,x2")
        for (pt, L, ℓ) in ell_scan
            println(io, join(ell_run(pt, L, ℓ), ",")); flush(io)
        end
    end
end

E = readdlm(ell_file, ','; header = true)[1]
@printf("%6s %11s %4s | %4s %4s | %7s | %12s %13s %7s\n", "point", "substrate", "ℓ", "χ_K", "χ_ψ", "DMRG", "E_f", "ΔQ", "gap")
for r in sort(collect(eachrow(E)); by = r -> (r[1], r[2], r[3]))
    @printf("%6s %5d×%-5d %4d | %4d %4d | %6.0fs | %12.4e %+13.4e %7.4f\n", r[1], 2^r[2], 2^r[2], r[3], r[5], r[6], r[9], r[11], r[12], r[13])
end

 point   substrate    ℓ |  χ_K  χ_ψ |    DMRG |          E_f            ΔQ     gap
 bench   512×512      8 |   38   26 |    243s |   4.3985e-04   +3.3762e-02  0.5586


 bench   512×512     12 |   38   24 |    395s |   3.7123e-04   -3.4466e-03  0.5529
 bench   512×512     16 |   36   29 |    439s |   2.3356e-05   +1.7294e-04  0.5511
 bench   512×512     24 |   38   24 |    487s |   2.6066e-07   -1.8332e-06  0.5499
 bench   512×512     32 |   37   24 |    399s |   1.7369e-09   -4.5961e-08  0.5494
 bench   512×512     48 |   38   24 |    880s |   3.1916e-14   -1.0447e-09  0.5491
 bench   512×512     64 |   34   24 |    507s |   2.3227e-14   +3.9062e-11  0.5489
 bench   512×512     96 |   38   27 |    555s |   9.4204e-16   +2.7201e-10  0.5489
 bench   512×512    128 |   31   27 |    455s |   1.2307e-13   +1.3518e-10  0.5489
 bench  1024×1024    64 |   37   24 |   2109s |   4.1460e-15   -1.2855e-10  0.5490
 bench  1024×1024   128 |   38   24 |   1177s |   1.7804e-15   -3.1646e-11  0.5490
  cand   512×512     16 |   38  124 |   2610s |   5.8327e-03   -5.3766e-02  0.1225
  cand   512×512     24 |   38  131 |   1520s |   3.3407e-04   +3.3341e-02  0.1036
  ca